# HarDNet-MSEG — tanítás és benchmark (SZE + opcionális Kvasir-SEG)

Ez a notebook a `colon_short_szd_r.ipynb` (DeepLabV3+ MobileNet) mintájára készült **HarDNet-MSEG** modellre.

## Lokális adatok (nem a repóban)

- `images/PI-program-JNET-classes.xlsx`, képek: `images/`, maszkok: `images/masks/`
- Opcionális: `Kvasir-SEG/` (trainhez, mint DeepLabnél)
- **HarDNet-68 előtanított backbone** (`hardnet68.pth`): [PingoLH/Pytorch-HarDNet](https://github.com/PingoLH/Pytorch-HarDNet)
  - `HARDNET68_PRETRAINED_PATH` környezeti változó **vagy** a modell-init cellában a konstans

## Checkpoint mappa

Csak **`checkpoints_hardnet/`** — ne írj a `checkpoints_0409/` vagy `checkpoints_unet/` mappákba.

## Upstream / licenc

HarDNet-MSEG: [james128333/HarDNet-MSEG](https://github.com/james128333/HarDNet-MSEG) — Apache-2.0 (`network/hardnet_mseg/`).

## Tanítás (Levente)

1. `pip install -r requirements_train.txt`
2. Futtasd a cellákat sorrendben; a **Tanítás** cellát csak akkor, ha az adatok és a `hardnet68.pth` útvonal rendben van.
3. Bemenet: **513×513** (DeepLab transformok).

**CI / ez a PR:** tanítás nem futott; import/szintaxis ellenőrzés céljából készült.


In [ ]:
import os
import sys
import json
import re
import glob as glob_module
import time
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils import data

import network
from network.hardnet_mseg import HarDMSEGTwoClass, HARDNET68_PRETRAINED_PATH_ENV
from utils import ext_transforms as et

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

RUN_DIR = "checkpoints_hardnet"
MODEL_TAG = "best_model_hardnet"
os.makedirs(RUN_DIR, exist_ok=True)


In [ ]:
def read_images(excel_path):
    classes = read_excel(excel_path)
    data = pd.DataFrame(columns=['ID', 'CLASS', 'IMAGE', 'QUALITY', 'SECOND_LETTER'])
    noimagecount = 0
    # Az images könyvtár elérési útja
    images_dir = os.path.dirname(excel_path)
    
    for index, row in classes.iterrows():
        image_id = row['ID']
        image_files = [f for f in os.listdir(images_dir) if f.lower().startswith(image_id.lower())]
        if image_files:
            for image in image_files:
                img_path = os.path.join(images_dir, image)
                img = cv2.imread(img_path)
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img_pil = Image.fromarray(img)
                    img_array = np.array(img_pil)
                    
                    # Determine quality based on filename
                    quality = image[6] if image[6] in ['E', 'R'] else None
                    
                    # Determine the second letter based on filename
                    second_letter = image[8] if len(image) > 8 else None
                    
                    data = pd.concat([data, pd.DataFrame([{'ID': image_id, 
                        'CLASS': row['CLASS'], 
                        'IMAGE': img_array, 
                        'QUALITY': quality,
                        'SECOND_LETTER': second_letter
                    }])], ignore_index=True)
        else:
            print("No image found for ID: ", image_id)
            noimagecount += 1
    print("Number of images not found: ", noimagecount)
    return data

def read_excel(excel_path):
    df = pd.read_excel(excel_path)
    columns = df.columns
    if 'Wrong' in df.columns:
        df = df[df['Wrong'] != 'x']
    new_df = pd.DataFrame(columns=['ID', 'CLASS'])

    new_df['ID'] = df['ID']
    
    # CLASS oszlop feltöltése a JNET kategóriák alapján
    for index, row in df.iterrows():
        class_value = None
        for jnet_class in ['JNET_1', 'JNET_2A', 'JNET_2B', 'JNET_3']:
            if jnet_class in df.columns and row[jnet_class] == 'x':
                class_value = jnet_class
                break
        new_df.loc[index, 'CLASS'] = class_value
    
    return new_df


In [ ]:
excel_path = "images/PI-program-JNET-classes.xlsx"

sze = read_images(excel_path)
sze.head()


In [ ]:
def prepare_sze_dataframe(sze, images_dir="images", masks_dir="images/masks"):
    # Create a new dataframe with the required columns
    sze_prepared = pd.DataFrame(columns=['ID', 'CLASS', 'image_file', 'image_path', 'mask_path', 'image_array', 'mask_array', 'image_height', 'image_width', 'mask_height', 'mask_width', 'polyp_area'])
    
    for index, row in sze.iterrows():
        # Helyes fájlnév összeállítása az ID, QUALITY és SECOND_LETTER alapján
        #print(row['ID'])
        image_file = f"{row['ID']}-{row['QUALITY']}-{row['SECOND_LETTER']}.jpg"
        mask_file = f"{row['ID']}-{row['QUALITY']}-{row['SECOND_LETTER']}-polyp.png"  # Helyes maszk fájlnév
        image_path = os.path.join(images_dir, image_file)
        mask_path = os.path.join(masks_dir, mask_file)
        
        # Check if image exists
        if not os.path.exists(image_path):
            print(f"Image not found: {image_path}")
            continue
        
        # Read the image with correct color order (BGR to RGB)
        image_array = cv2.imread(image_path)
        if image_array is None:
            print(f"Failed to read image: {image_path}")
            continue
        
        # Convert BGR to RGB to fix the blue tint issue
        image_array = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
        
        image_height, image_width = image_array.shape[:2]
        
        # Read the mask
        if os.path.exists(mask_path):
            mask_array = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask_array is None:
                print(f"Failed to read mask: {mask_path}")
                mask_array = np.zeros((image_height, image_width), dtype=np.uint8)
        else:
            print(f"Mask not found: {mask_path}")
            mask_array = np.zeros((image_height, image_width), dtype=np.uint8)
        
        # Calculate polyp area (sum of mask pixels)
        mask_height, mask_width = mask_array.shape[:2] if mask_array is not None else (0, 0)
        polyp_area = np.sum(mask_array) if mask_array is not None else 0
        
        # Add the row to the new dataframe
        sze_prepared = pd.concat([sze_prepared, pd.DataFrame([{'ID': row['ID'],
            'CLASS': row['CLASS'],
            'image_file': image_file,
            'image_path': image_path,
            'mask_path': mask_path,
            'image_array': image_array,
            'mask_array': mask_array,
            'image_height': image_height,
            'image_width': image_width,
            'mask_height': mask_height,
            'mask_width': mask_width,
            'polyp_area': polyp_area
        }])], ignore_index=True)
    
    return sze_prepared

sze_prepared = prepare_sze_dataframe(sze, images_dir="images", masks_dir="images/masks")
sze_prepared.head()


In [ ]:
# 🔧 ADATTRANSZFORMÁCIÓK DEFINIÁLÁSA
# Ez a FŐDEFINÍCIÓ - itt definiáljuk az összes szükséges transzformációt

def get_train_transform():
    """
    Tréning adatok transzformációja augmentációval:
    - Véletlenszerű skálázás 0.5x és 2.0x között
    - Véletlenszerű kivágás 513x513 méretben
    - Véletlenszerű vízszintes tükrözés
    - Tenzorrá alakítás és normalizálás
    """
    train_transform = et.ExtCompose([
        et.ExtRandomScale((0.5, 2.0)),
        et.ExtRandomCrop(size=(513, 513), pad_if_needed=True),
        et.ExtRandomHorizontalFlip(),
        et.ExtToTensor(),
        et.ExtNormalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform

def get_val_transform():
    """
    Validációs/teszt adatok transzformációja (augmentáció nélkül):
    - Átméretezés 513 pixelre
    - Középső kivágás
    - Tenzorrá alakítás és normalizálás
    """
    val_transform = et.ExtCompose([
        et.ExtResize(size=513),
        et.ExtCenterCrop(size=513),
        et.ExtToTensor(),
        et.ExtNormalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return val_transform

print("✅ Transform függvények definiálva!")


In [ ]:
class KvasirSEGDataset(data.Dataset):
    """
    Kvasir-SEG adatkészlet polip szegmentációhoz - EGYSÉGES DEFINÍCIÓ
    Támogatja mind a DataFrame-ben tárolt tömböket, mind a fájl útvonalakat
    """
    # Színtérkép definiálása
    cmap = np.zeros((256, 3), dtype=np.uint8)
    cmap[0] = [0, 0, 0]      # Háttér - fekete
    cmap[1] = [255, 255, 255]  # Polip - fehér

    def __init__(self, dataframe, transform=None, img_col='image_path', mask_col='mask_path', file_col=None):
        """
        Paraméterek:
            dataframe: Pandas DataFrame képekkel és maszkokkal
            transform: Alkalmazandó transzformáció
            img_col: Kép oszlop neve (lehet 'image_path' vagy 'image_array')
            mask_col: Maszk oszlop neve (lehet 'mask_path' vagy 'mask_array') 
            file_col: Fájlnév oszlop neve (opcionális)
        """
        self.df = dataframe
        self.transform = transform
        self.img_col = img_col
        self.mask_col = mask_col
        self.file_col = file_col

    def __getitem__(self, index):
        """
        Paraméterek:
            index (int): Index
        Visszatérési érték:
            tuple: (kép, maszk) vagy (kép, maszk, fájlnév)
        """
        # Fájlnév megszerzése, ha van
        filename = None
        if self.file_col and self.file_col in self.df.columns:
            filename = self.df.iloc[index][self.file_col]
        
        # Kép betöltése - ellenőrizzük, hogy tömb vagy útvonal-e
        if self.img_col == 'image_array' and 'image_array' in self.df.columns:
            # DataFrame-ben tárolt tömb
            img_array = self.df.iloc[index]['image_array']
            img = Image.fromarray(img_array)
        else:
            # Fájl útvonal
            img_path = self.df.iloc[index][self.img_col]
            if not os.path.exists(img_path):
                print(f"FIGYELEM: Kép nem található: {img_path}")
                # Helyettesítő kép létrehozása
                img = Image.new('RGB', (256, 256), color=(128, 128, 128))
                if filename is None:
                    filename = os.path.basename(img_path)
            else:
                img = Image.open(img_path).convert('RGB')
                if filename is None:
                    filename = os.path.basename(img_path)
        
        # Maszk betöltése - ellenőrizzük, hogy tömb vagy útvonal-e
        if self.mask_col == 'mask_array' and 'mask_array' in self.df.columns:
            # DataFrame-ben tárolt tömb
            mask_array = self.df.iloc[index]['mask_array']
            if mask_array is not None:
                mask = Image.fromarray(mask_array)
            else:
                mask = Image.new('L', img.size, 0)
        else:
            # Fájl útvonal
            mask_path = self.df.iloc[index][self.mask_col]
            if os.path.exists(mask_path):
                mask = Image.open(mask_path).convert('L')  # Szürkeárnyalatos
                # Binarizálás
                mask = np.array(mask)
                mask = (mask > 128).astype(np.uint8)
                mask = Image.fromarray(mask)
            else:
                # Üres maszk létrehozása
                mask = Image.new('L', img.size, 0)
        
        # Transzformációk alkalmazása
        if self.transform is not None:
            img, mask = self.transform(img, mask)
        
        # Visszatérés fájlnévvel vagy anélkül
        if filename is not None:
            return img, mask, filename
        else:
            return img, mask

    def __len__(self):
        return len(self.df)

    @classmethod
    def decode_target(cls, mask):
        """Szemantikus maszk dekódolása RGB képpé"""
        return cls.cmap[mask]


In [ ]:
# --- Adathalmazok Betöltése ---

# 1. Kvasir-SEG adathalmaz betöltése
kvasir_root = "Kvasir-SEG"
kvasir_images_dir = os.path.join(kvasir_root, "")
kvasir_masks_dir = os.path.join(kvasir_root, "masks")

kvasir_files = [f for f in os.listdir(kvasir_images_dir) if f.endswith('.jpg') or f.endswith('.png')]
kvasir_data = []

for img_file in kvasir_files:
    img_path = os.path.join(kvasir_images_dir, img_file)
    mask_path = os.path.join(kvasir_masks_dir, img_file)
    
    # Maszkfájl ellenőrzése alternatív kiterjesztésekkel
    if not os.path.exists(mask_path):
        base_name, _ = os.path.splitext(img_file)
        for ext in ['.jpg', '.png', '.tif']:
            alt_mask_path = os.path.join(kvasir_masks_dir, base_name + ext)
            if os.path.exists(alt_mask_path):
                mask_path = alt_mask_path
                break
    
    kvasir_data.append({'ID': 'kvasir_' + os.path.splitext(img_file)[0], # Egyedi ID generálása
        'CLASS': 1, # A Kvasir-SEG minden képe tartalmaz polipot
        'image_path': img_path,
        'mask_path': mask_path
    })

df_kvasir = pd.DataFrame(kvasir_data)
print(f"Kvasir-SEG adathalmaz betöltve: {len(df_kvasir)} kép.")

# 2. SZE adathalmaz használata
# A `sze_prepared` DataFrame-et használjuk, amit a korábbi cellákban hoztunk létre.
try:
    if 'sze_prepared' in locals() and not sze_prepared.empty:
        df_sze = sze_prepared.copy()
        # A 'CLASS' oszlop binárissá alakítása (0 vagy 1)
        # 1, ha a CLASS nem None és nem 0, egyébként 0
        df_sze['CLASS'] = df_sze['CLASS'].apply(lambda x: 1 if pd.notna(x) and x != 0 else 0)
        print(f"SZE adathalmaz betöltve: {len(df_sze)} kép.")
        print("SZE CLASS eloszlás:")
        print(df_sze['CLASS'].value_counts())
    else:
        raise NameError("sze_prepared DataFrame nem létezik vagy üres.")
except NameError as e:
    print(f"Hiba: {e}. Futtasd a korábbi cellákat a SZE adathalmaz létrehozásához.")
    df_sze = pd.DataFrame() # Üres DataFrame hibamegelőzésként


### Adatok felosztása train/validation/test (fix patient-ID listák)

Ugyanazok a listák, mint a DeepLab notebookban.


In [ ]:
import numpy as np
import pandas as pd

# --- Fix ID-listák a train/validation/test felosztáshoz ---

val_ids = [
    'PI007', 'PI013', 'PI014', 'PI023', 'PI033', 'PI036', 'PI047', 'PI053', 'PI056',
    'PI061', 'PI065', 'PI074', 'PI077', 'PI082', 'PI084', 'PI090', 'PI097', 'PI108',
    'PI110', 'PI111', 'PI133', 'PI153', 'PI156', 'PI170', 'PI172', 'PI193', 'PI194',
    'PI196', 'PI206', 'PI209', 'PI212', 'PI213', 'PI227', 'PI229', 'PI239', 'PI254',
    'PI257', 'PI262', 'PI263', 'PI265', 'PI269', 'PI284', 'PI288', 'PI293', 'PI300',
    'PI302', 'PI305', 'PI311', 'PI319', 'PI321', 'PI325', 'PI336', 'PI347', 'PI357',
    'PI358', 'PI366', 'PI369', 'PI373', 'PI380', 'PI382', 'PI386', 'PI387', 'PI396',
    'PI398', 'PI406', 'PI412', 'PI413', 'PI416', 'PI420', 'PI431', 'PI447', 'PI448',
    'PI451', 'PI455', 'PI464', 'PI467', 'PI482', 'PI484', 'PI488',
]

test_ids = [
    'PI009', 'PI015', 'PI016', 'PI037', 'PI048', 'PI050', 'PI055', 'PI057', 'PI068',
    'PI072', 'PI073', 'PI078', 'PI080', 'PI085', 'PI086', 'PI091', 'PI112', 'PI113',
    'PI114', 'PI121', 'PI123', 'PI130', 'PI135', 'PI136', 'PI137', 'PI142', 'PI149',
    'PI152', 'PI166', 'PI171', 'PI175', 'PI178', 'PI181', 'PI184', 'PI187', 'PI189',
    'PI191', 'PI197', 'PI198', 'PI204', 'PI205', 'PI207', 'PI210', 'PI211', 'PI219',
    'PI224', 'PI230', 'PI238', 'PI243', 'PI247', 'PI249', 'PI252', 'PI261', 'PI266',
    'PI270', 'PI272', 'PI277', 'PI280', 'PI290', 'PI298', 'PI310', 'PI335', 'PI338',
    'PI348', 'PI349', 'PI367', 'PI368', 'PI370', 'PI372', 'PI375', 'PI376', 'PI389',
    'PI390', 'PI399', 'PI400', 'PI405', 'PI408', 'PI415', 'PI418', 'PI419', 'PI432',
    'PI433', 'PI445', 'PI454', 'PI456', 'PI462', 'PI468', 'PI472', 'PI473', 'PI478',
    'PI480',
]

train_ids = [
    'PI003', 'PI005', 'PI006', 'PI008', 'PI011', 'PI017', 'PI018', 'PI019', 'PI020',
    'PI021', 'PI022', 'PI024', 'PI025', 'PI026', 'PI028', 'PI029', 'PI030', 'PI031',
    'PI032', 'PI034', 'PI035', 'PI038', 'PI039', 'PI040', 'PI041', 'PI042', 'PI043',
    'PI045', 'PI046', 'PI049', 'PI051', 'PI052', 'PI058', 'PI059', 'PI060', 'PI062',
    'PI063', 'PI066', 'PI067', 'PI069', 'PI070', 'PI071', 'PI075', 'PI076', 'PI079',
    'PI081', 'PI083', 'PI087', 'PI088', 'PI089', 'PI092', 'PI093', 'PI094', 'PI095',
    'PI096', 'PI098', 'PI099', 'PI100', 'PI102', 'PI103', 'PI104', 'PI105', 'PI107',
    'PI109', 'PI115', 'PI116', 'PI118', 'PI119', 'PI120', 'PI122', 'PI124', 'PI125',
    'PI126', 'PI127', 'PI129', 'PI132', 'PI134', 'PI138', 'PI139', 'PI141', 'PI143',
    'PI144', 'PI145', 'PI146', 'PI147', 'PI148', 'PI150', 'PI151', 'PI155', 'PI157',
    'PI158', 'PI159', 'PI160', 'PI161', 'PI162', 'PI163', 'PI164', 'PI165', 'PI168',
    'PI169', 'PI174', 'PI176', 'PI177', 'PI179', 'PI180', 'PI182', 'PI185', 'PI188',
    'PI190', 'PI200', 'PI201', 'PI202', 'PI203', 'PI208', 'PI214', 'PI215', 'PI216',
    'PI217', 'PI218', 'PI221', 'PI222', 'PI225', 'PI226', 'PI228', 'PI231', 'PI232',
    'PI233', 'PI234', 'PI235', 'PI236', 'PI237', 'PI240', 'PI241', 'PI242', 'PI244',
    'PI245', 'PI246', 'PI248', 'PI250', 'PI251', 'PI253', 'PI255', 'PI256', 'PI259',
    'PI260', 'PI264', 'PI268', 'PI271', 'PI273', 'PI274', 'PI275', 'PI278', 'PI279',
    'PI281', 'PI282', 'PI283', 'PI285', 'PI291', 'PI292', 'PI294', 'PI295', 'PI296',
    'PI297', 'PI303', 'PI304', 'PI306', 'PI307', 'PI308', 'PI312', 'PI313', 'PI314',
    'PI317', 'PI318', 'PI320', 'PI322', 'PI323', 'PI327', 'PI328', 'PI330', 'PI331',
    'PI332', 'PI333', 'PI334', 'PI337', 'PI339', 'PI340', 'PI341', 'PI342', 'PI343',
    'PI344', 'PI345', 'PI350', 'PI351', 'PI352', 'PI353', 'PI354', 'PI359', 'PI360',
    'PI362', 'PI363', 'PI365', 'PI371', 'PI374', 'PI377', 'PI378', 'PI379', 'PI381',
    'PI383', 'PI384', 'PI391', 'PI392', 'PI393', 'PI394', 'PI395', 'PI397', 'PI402',
    'PI403', 'PI404', 'PI407', 'PI409', 'PI411', 'PI414', 'PI417', 'PI421', 'PI422',
    'PI423', 'PI424', 'PI425', 'PI426', 'PI427', 'PI428', 'PI429', 'PI430', 'PI434',
    'PI435', 'PI436', 'PI437', 'PI438', 'PI439', 'PI440', 'PI441', 'PI442', 'PI443',
    'PI444', 'PI446', 'PI449', 'PI452', 'PI453', 'PI457', 'PI458', 'PI459', 'PI460',
    'PI461', 'PI463', 'PI465', 'PI466', 'PI469', 'PI470', 'PI471', 'PI474', 'PI475',
    'PI477', 'PI479', 'PI481', 'PI483', 'PI485', 'PI486', 'PI487', 'PI491',
]

# Nagybetűsre alakítás az összehasonlításhoz (pl. pi205 -> PI205)
val_ids_upper = [x.upper() for x in val_ids]
test_ids_upper = [x.upper() for x in test_ids]
train_ids_upper = [x.upper() for x in train_ids]

# --- SZE Adathalmaz Felosztása fix ID-listák alapján ---

try:
    df_sze_upper_id = df_sze['ID'].str.upper()

    sze_train = df_sze[df_sze_upper_id.isin(train_ids_upper)].reset_index(drop=True)
    sze_val = df_sze[df_sze_upper_id.isin(val_ids_upper)].reset_index(drop=True)
    sze_test = df_sze[df_sze_upper_id.isin(test_ids_upper)].reset_index(drop=True)

    print("--- Fix ID-lista Alapú Felosztás Eredménye (SZE adatok) ---")
    print(f"Train halmaz (SZE): {sze_train.shape} (Páciens ID-k: {len(sze_train['ID'].str.upper().unique())} db)")
    print(f"Validation halmaz (SZE): {sze_val.shape} (Páciens ID-k: {len(sze_val['ID'].str.upper().unique())} db)")
    print(f"Test halmaz (SZE): {sze_test.shape} (Páciens ID-k: {len(sze_test['ID'].str.upper().unique())} db)")

    # Kvasir adathalmaz hozzáadása a SZE tanító adatokhoz
    train_df = pd.concat([sze_train, df_kvasir], ignore_index=True)
    val_df = sze_val
    test_df = sze_test

    # Eredmények kiírása
    print("\n--- Végső Adathalmazok Mérete ---")
    print(f"Teljes tanító adathalmaz (SZE + Kvasir): {len(train_df)} kép")
    print(f"Validációs adathalmaz (csak SZE): {len(val_df)} kép")
    print(f"Teszt adathalmaz (csak SZE): {len(test_df)} kép")

except NameError as e:
    print(f"Hiba: {e}. Ellenőrizd, hogy a `df_sze` és `df_kvasir` DataFrame-ek léteznek-e az előző cella futtatása után.")
except Exception as e:
    print(f"Váratlan hiba történt a felosztás során: {e}")


In [ ]:
def verify_split(sze_train, sze_val, sze_test, train_ids, val_ids, test_ids):
    """
    Részletes ellenőrzés a fix ID-lista alapú split után:
    1. Képszám ellenőrzés (elvárt vs tényleges)
    2. ID-k helyesség ellenőrzés (minden megadott ID megtalálható-e, van-e felesleges)
    3. Átfedés ellenőrzés (data leakage)
    """
    all_ok = True

    # --- Nagybetűs ID halmazok ---
    expected_train = set(x.upper() for x in train_ids)
    expected_val = set(x.upper() for x in val_ids)
    expected_test = set(x.upper() for x in test_ids)

    actual_train = set(sze_train['ID'].str.upper().unique())
    actual_val = set(sze_val['ID'].str.upper().unique())
    actual_test = set(sze_test['ID'].str.upper().unique())

    # === 1. Képszám ellenőrzés ===
    print("=" * 60)
    print("1. KÉPSZÁM ELLENŐRZÉS")
    print("=" * 60)

    expected_counts = {'Train (SZE)': 678, 'Validation': 189, 'Test': 206}
    actual_counts = {
        'Train (SZE)': len(sze_train),
        'Validation': len(sze_val),
        'Test': len(sze_test),
    }
    for name in expected_counts:
        exp = expected_counts[name]
        act = actual_counts[name]
        status = "OK" if act == exp else "ELTÉRÉS"
        if act != exp:
            all_ok = False
        print(f"  {name}: elvárt={exp}, tényleges={act}  [{status}]")

    # === 2. ID-k helyesség ellenőrzés ===
    print("\n" + "=" * 60)
    print("2. ID-K HELYESSÉG ELLENŐRZÉS")
    print("=" * 60)

    for name, expected, actual in [
        ('Train', expected_train, actual_train),
        ('Validation', expected_val, actual_val),
        ('Test', expected_test, actual_test),
    ]:
        missing = expected - actual
        extra = actual - expected
        print(f"\n  {name}:")
        print(f"    Elvárt ID-k száma: {len(expected)}, Tényleges: {len(actual)}")
        if missing:
            all_ok = False
            print(f"    HIÁNYZÓ ID-k ({len(missing)} db): {sorted(missing)}")
        else:
            print(f"    Minden elvárt ID megtalálható. [OK]")
        if extra:
            all_ok = False
            print(f"    FELESLEGES ID-k ({len(extra)} db): {sorted(extra)}")
        else:
            print(f"    Nincs felesleges ID. [OK]")

    # === 3. Átfedés ellenőrzés (data leakage) ===
    print("\n" + "=" * 60)
    print("3. ÁTFEDÉS ELLENŐRZÉS (DATA LEAKAGE)")
    print("=" * 60)

    train_val_overlap = actual_train & actual_val
    train_test_overlap = actual_train & actual_test
    val_test_overlap = actual_val & actual_test

    if train_val_overlap:
        all_ok = False
        print(f"  HIBA - Train-Val átfedés ({len(train_val_overlap)} db): {sorted(train_val_overlap)}")
    else:
        print(f"  Train-Val átfedés: nincs [OK]")

    if train_test_overlap:
        all_ok = False
        print(f"  HIBA - Train-Test átfedés ({len(train_test_overlap)} db): {sorted(train_test_overlap)}")
    else:
        print(f"  Train-Test átfedés: nincs [OK]")

    if val_test_overlap:
        all_ok = False
        print(f"  HIBA - Val-Test átfedés ({len(val_test_overlap)} db): {sorted(val_test_overlap)}")
    else:
        print(f"  Val-Test átfedés: nincs [OK]")

    # === Összefoglaló ===
    print("\n" + "=" * 60)
    if all_ok:
        print("ÖSSZEFOGLALÓ: MINDEN RENDBEN - a split megfelel az elvártnak.")
    else:
        print("ÖSSZEFOGLALÓ: HIBÁK TALÁLHATÓK - ellenőrizd a fenti részleteket!")
    print("=" * 60)

    return all_ok

verify_split(sze_train, sze_val, sze_test, train_ids, val_ids, test_ids)


In [ ]:
# Adat Augmentáció
augmentation_multiplier = 3

def multiply_dataframe_rows(df, multiplier):
    if multiplier <= 1:
        return df.copy()
    return pd.concat([df.copy() for _ in range(multiplier)], ignore_index=True)

train_df_augmented = multiply_dataframe_rows(train_df, augmentation_multiplier)
val_df_augmented = multiply_dataframe_rows(val_df, augmentation_multiplier)

print(f"Eredeti train méret: {len(train_df)}, Augmentált: {len(train_df_augmented)}")
print(f"Eredeti val méret: {len(val_df)}, Augmentált: {len(val_df_augmented)}")



In [ ]:
# --- HarDNet-MSEG modell ---
# Töltsd ki a hardnet68.pth abszolút útvonalát (vagy export HARDNET68_PRETRAINED_PATH).

HARDNET68_PRETRAINED_PATH = os.environ.get(HARDNET68_PRETRAINED_PATH_ENV, "")
# Példa Windows: r"C:\\models\\hardnet68.pth"

_load_backbone = bool(HARDNET68_PRETRAINED_PATH) and os.path.isfile(HARDNET68_PRETRAINED_PATH)
if HARDNET68_PRETRAINED_PATH and not _load_backbone:
    print(f"FIGYELEM: HARDNET68_PRETRAINED_PATH megadva, de a fájl nem létezik: {HARDNET68_PRETRAINED_PATH}")

model = HarDMSEGTwoClass(
    pretrained_backbone=_load_backbone,
    backbone_weight_path=HARDNET68_PRETRAINED_PATH or None,
).to(device)

candidate_files = sorted(
    glob_module.glob(os.path.join(RUN_DIR, f"{MODEL_TAG}_epoch_*.pth")),
    key=lambda f: int(re.search(r"epoch_(\d+)", f).group(1)),
)
if candidate_files:
    save_path = candidate_files[-1]
    print(f"Meglévő HarDNet checkpoint betöltése: {save_path}")
    ckpt = torch.load(save_path, map_location=device, weights_only=False)
    state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    model.load_state_dict(state, strict=False)
else:
    print("Nincs korábbi HarDNet checkpoint — csak backbone (ha megadva) + új fej.")


In [ ]:
# --- ADATKÉSZLETEK ÉS ADATBETÖLTŐK LÉTREHOZÁSA ---

train_transform = get_train_transform()
val_transform = get_val_transform()
test_transform = get_val_transform()

train_dataset = KvasirSEGDataset(dataframe=train_df_augmented, transform=train_transform)
val_dataset = KvasirSEGDataset(dataframe=val_df_augmented, transform=val_transform)
test_dataset = KvasirSEGDataset(dataframe=test_df, transform=test_transform)

batch_size = 4
num_workers = 0

train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print("Adatbetöltők sikeresen létrehozva.")

INPUT_SIZE = (513, 513)  # train/val crop — latency benchmark is ezt használja


In [ ]:
# Metrikák, tanító és validációs függvények
def calculate_metrics(outputs, targets):
    preds = outputs.argmax(1)
    pred_polyp = (preds == 1)
    target_polyp = (targets == 1)
    intersection = (pred_polyp & target_polyp).sum().float()
    union = (pred_polyp | target_polyp).sum().float()
    pred_area = pred_polyp.sum().float()
    target_area = target_polyp.sum().float()
    dice = 2 * intersection / (pred_area + target_area + 1e-6)
    iou = intersection / (union + 1e-6)
    return {'dice': dice.item(), 'iou': iou.item()}

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, sample_count, dice_sum, iou_sum = 0.0, 0, 0.0, 0.0
    for batch in tqdm(loader, desc="Training"):
        images, targets = batch[0], batch[1]
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets.long())
        loss.backward()
        optimizer.step()
        
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        sample_count += batch_size
        metrics = calculate_metrics(outputs, targets)
        dice_sum += metrics['dice'] * batch_size
        iou_sum += metrics['iou'] * batch_size
    return {'loss': running_loss/sample_count, 'dice': dice_sum/sample_count, 'iou': iou_sum/sample_count}

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, sample_count, dice_sum, iou_sum = 0.0, 0, 0.0, 0.0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation"):
            images, targets = batch[0], batch[1]
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets.long())

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            sample_count += batch_size
            metrics = calculate_metrics(outputs, targets)
            dice_sum += metrics['dice'] * batch_size
            iou_sum += metrics['iou'] * batch_size
    return {'val_loss': running_loss/sample_count, 'val_dice': dice_sum/sample_count, 'val_iou': iou_sum/sample_count}


In [ ]:
# --- Tanítás (futtatás opcionális — Levente) ---
RUN_TRAINING = False  # Állítsd True-ra lokálisan a tanítás indításához

TOTAL_EPOCHS = 50
EARLY_STOP_PATIENCE = 7

start_epoch = 0
best_iou = 0.0
history = {
    "train_loss": [], "train_dice": [], "train_iou": [],
    "val_loss": [], "val_dice": [], "val_iou": [],
    "lr": [],
}

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.1, patience=3)

checkpoint_files = sorted(
    glob_module.glob(os.path.join(RUN_DIR, f"{MODEL_TAG}_epoch_*.pth")),
    key=lambda f: int(re.search(r"epoch_(\d+)", f).group(1)),
)
if checkpoint_files:
    latest_ckpt = checkpoint_files[-1]
    ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
    if "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = int(ckpt.get("epoch", -1)) + 1
    best_iou = float(ckpt.get("iou", 0.0))
    if "history" in ckpt:
        history = ckpt["history"]
    print(f"Folytatás epoch {start_epoch + 1}-től, eddigi best IoU: {best_iou:.4f}")

if RUN_TRAINING:
    epochs_without_improvement = 0
    for epoch in range(start_epoch, TOTAL_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{TOTAL_EPOCHS}")
        train_metrics = train_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = validate(model, val_loader, criterion, device)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_metrics["loss"])
        history["train_dice"].append(train_metrics["dice"])
        history["train_iou"].append(train_metrics["iou"])
        history["val_loss"].append(val_metrics["val_loss"])
        history["val_dice"].append(val_metrics["val_dice"])
        history["val_iou"].append(val_metrics["val_iou"])
        history["lr"].append(current_lr)

        print(
            f"Train Loss: {train_metrics['loss']:.4f}, Dice: {train_metrics['dice']:.4f}, IoU: {train_metrics['iou']:.4f}"
        )
        print(
            f"Val Loss: {val_metrics['val_loss']:.4f}, Dice: {val_metrics['val_dice']:.4f}, IoU: {val_metrics['val_iou']:.4f}"
        )

        scheduler.step(val_metrics["val_iou"])

        if val_metrics["val_iou"] > best_iou:
            best_iou = val_metrics["val_iou"]
            epochs_without_improvement = 0
            save_path = os.path.join(RUN_DIR, f"{MODEL_TAG}_epoch_{epoch + 1}.pth")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "iou": best_iou,
                    "history": history,
                    "checkpoint_dir": RUN_DIR,
                    "model_tag": MODEL_TAG,
                    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                },
                save_path,
            )
            print(f"Saved new best model to {save_path}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"Early stopping. Legjobb Val IoU: {best_iou:.4f}")
            break

    pd.DataFrame(history).to_csv(os.path.join(RUN_DIR, "training_history_hardnet.csv"), index=False)
    with open(os.path.join(RUN_DIR, "training_history_hardnet.json"), "w") as f:
        json.dump(history, f, indent=4)
    print(f"Tanítás kész. Checkpointok: {RUN_DIR}")
else:
    print("RUN_TRAINING=False — tanítás kihagyva (alapértelmezett PR/CI mód).")


In [ ]:
def plot_training_history(history, run_dir=RUN_DIR):
    if not history.get("train_loss"):
        print("Nincs history — futtasd a tanítást vagy tölts be checkpointot history-val.")
        return
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], "b-", label="Training Loss")
    plt.plot(epochs, history["val_loss"], "r-", label="Validation Loss")
    plt.title("Loss")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_iou"], "b-", label="Training IoU")
    plt.plot(epochs, history["val_iou"], "r-", label="Validation IoU")
    plt.title("IoU")
    plt.legend()
    os.makedirs(run_dir, exist_ok=True)
    plt.savefig(os.path.join(run_dir, "training_curves_hardnet.png"), bbox_inches="tight", dpi=200)
    plt.show()

# plot_training_history(history)


In [ ]:
def predict_and_save_masks(model, test_loader, model_name="hardnet_mseg", device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    root_dir = "images"
    model_dir = os.path.join(root_dir, model_name)
    masks_dir = os.path.join(model_dir, "masks")
    polyp_dir = os.path.join(model_dir, "polyp")
    os.makedirs(masks_dir, exist_ok=True)
    os.makedirs(polyp_dir, exist_ok=True)

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def unnormalize(tensor):
        x = tensor.detach().cpu().permute(1, 2, 0).numpy().astype(np.float32)
        x = x * std + mean
        return np.clip(x, 0.0, 1.0)

    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            if len(batch) == 3:
                images, _, filenames = batch
            else:
                images, _ = batch
                filenames = None
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            for i, pred in enumerate(preds):
                if filenames is None:
                    continue
                filename = filenames[i]
                base_filename = os.path.splitext(os.path.basename(filename))[0]
                if str(base_filename).lower() == "nan":
                    continue
                Image.fromarray((pred * 255).astype(np.uint8)).save(
                    os.path.join(masks_dir, f"{base_filename}_mask.png")
                )
                orig_img_np = unnormalize(images[i])
                final_img = np.zeros_like(orig_img_np)
                polyp_mask_rgb = np.stack([pred] * 3, axis=-1).astype(bool)
                final_img[polyp_mask_rgb] = orig_img_np[polyp_mask_rgb]
                Image.fromarray((final_img * 255).astype(np.uint8)).save(
                    os.path.join(polyp_dir, f"{base_filename}_polyp.png")
                )

# predict_and_save_masks(model, test_loader)


## Benchmark: Dice / IoU + latency

Ugyanaz a `calculate_metrics` stílus (2 osztályos logits), mint DeepLabnél. Baseline checkpointok opcionálisak; hiányuk esetén kihagyás.

- DeepLab: `checkpoints_0409/best_model_0409_epoch_31.pth`
- U-Net: `checkpoints_unet/best_model_epoch_22.pth`
- HarDNet: `checkpoints_hardnet/best_model_hardnet_epoch_*.pth`

Latency: batch=1, warm-up, mean/p50/p95 ms, bemenet **3×513×513**.


In [ ]:
import segmentation_models_pytorch as smp

RUN_BENCHMARK = False

DEEPLAB_CKPT = "checkpoints_0409/best_model_0409_epoch_31.pth"
UNET_CKPT = "checkpoints_unet/best_model_epoch_22.pth"
HARDNET_CKPT_GLOB = os.path.join(RUN_DIR, f"{MODEL_TAG}_epoch_*.pth")

LATENCY_INPUT_SIZE = (3, INPUT_SIZE[0], INPUT_SIZE[1])
LATENCY_WARMUP = 10
LATENCY_REPEATS = 50


def get_state_dict_from_checkpoint(ckpt_obj):
    if "model_state_dict" in ckpt_obj:
        return ckpt_obj["model_state_dict"]
    if "state_dict" in ckpt_obj:
        return ckpt_obj["state_dict"]
    return ckpt_obj


def evaluate_segmentation_model(model, loader, metrics_fn=calculate_metrics):
    model.eval()
    dice_sum, iou_sum, n = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            images, targets = batch[0], batch[1]
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            m = metrics_fn(outputs, targets)
            bs = images.size(0)
            dice_sum += m["dice"] * bs
            iou_sum += m["iou"] * bs
            n += bs
    return {"dice": dice_sum / max(n, 1), "iou": iou_sum / max(n, 1)}


def metrics_unet_binary(outputs, targets):
    probs = torch.sigmoid(outputs)
    preds = (probs > 0.5).float()
    if targets.ndim == 4:
        target_binary = (targets > 0.5).float()
    else:
        target_binary = (targets > 0.5).float().unsqueeze(1)
    intersection = (preds * target_binary).sum()
    pred_area = preds.sum()
    target_area = target_binary.sum()
    union = pred_area + target_area - intersection
    dice = (2 * intersection / (pred_area + target_area + 1e-6)).item()
    iou = (intersection / (union + 1e-6)).item()
    return {"dice": dice, "iou": iou}


def measure_latency_ms(model, input_size=LATENCY_INPUT_SIZE, warmup=LATENCY_WARMUP, repeats=LATENCY_REPEATS):
    model.eval()
    x = torch.randn(1, *input_size, device=device)
    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times = []
        for _ in range(repeats):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000.0)
    arr = np.array(times)
    return {
        "mean_ms": float(arr.mean()),
        "p50_ms": float(np.percentile(arr, 50)),
        "p95_ms": float(np.percentile(arr, 95)),
        "input_size": input_size,
        "device": str(device),
    }


def load_checkpoint_if_exists(path, build_fn, label):
    if not path or not os.path.isfile(path):
        print(f"[SKIP] {label}: nincs súlyfájl ({path})")
        return None
    m = build_fn().to(device)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    sd = get_state_dict_from_checkpoint(ckpt)
    m.load_state_dict(sd, strict=False)
    print(f"[OK] {label} betöltve: {path}")
    return m


def pick_latest_hardnet_ckpt():
    files = sorted(
        glob_module.glob(HARDNET_CKPT_GLOB),
        key=lambda f: int(re.search(r"epoch_(\d+)", f).group(1)),
    )
    return files[-1] if files else None


if RUN_BENCHMARK:
    benchmark_rows = []

    deeplab_m = load_checkpoint_if_exists(
        DEEPLAB_CKPT,
        lambda: network.modeling.__dict__["deeplabv3plus_mobilenet"](num_classes=2, output_stride=16),
        "DeepLabV3+ MobileNet",
    )
    if deeplab_m is not None:
        for split_name, loader in [("val", val_loader), ("test", test_loader)]:
            metrics = evaluate_segmentation_model(deeplab_m, loader)
            benchmark_rows.append({"model": "deeplab", "split": split_name, **metrics})
        benchmark_rows.append({"model": "deeplab", "split": "latency", **measure_latency_ms(deeplab_m)})

    unet_m = load_checkpoint_if_exists(
        UNET_CKPT,
        lambda: smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1),
        "U-Net ResNet34",
    )
    if unet_m is not None:
        for split_name, loader in [("val", val_loader), ("test", test_loader)]:
            metrics = evaluate_segmentation_model(unet_m, loader, metrics_fn=metrics_unet_binary)
            benchmark_rows.append({"model": "unet", "split": split_name, **metrics})
        benchmark_rows.append({"model": "unet", "split": "latency", **measure_latency_ms(unet_m)})

    hardnet_path = pick_latest_hardnet_ckpt()
    hardnet_m = load_checkpoint_if_exists(
        hardnet_path,
        lambda: HarDMSEGTwoClass(pretrained_backbone=False),
        "HarDNet-MSEG",
    )
    if hardnet_m is not None:
        for split_name, loader in [("val", val_loader), ("test", test_loader)]:
            metrics = evaluate_segmentation_model(hardnet_m, loader)
            benchmark_rows.append({"model": "hardnet_mseg", "split": split_name, **metrics})
        benchmark_rows.append({"model": "hardnet_mseg", "split": "latency", **measure_latency_ms(hardnet_m)})

    if benchmark_rows:
        df_bench = pd.DataFrame(benchmark_rows)
        out_csv = os.path.join(RUN_DIR, "benchmark_results.csv")
        df_bench.to_csv(out_csv, index=False)
        print(df_bench)
        print(f"Benchmark mentve: {out_csv}")
    else:
        print("Nincs betöltött modell — benchmark üres.")
else:
    print("RUN_BENCHMARK=False — benchmark kihagyva. Állítsd True-ra lokálisan.")
